# <span style="color:#1F4E79;">Synthetic CDM Parameter Recovery</span>

### Reproducing Figure 1(a) · Seshadri, Peysakhovich & Ugander (2019)

*Discovering Context Effects from Raw Choice Data*

Study how accurately a full Context-Dependent Model recovers known
interaction parameters as the number of observed choices increases.

**Workflow:** Generate parameters → Simulate choices → Estimate parameters
→ Measure recovery error → Compare across sample sizes.

Paper-specified settings and reproduction assumptions are documented separately.

## <span style="color:#1F4E79;">1. Setup</span>

Import the libraries and existing CDM package used throughout the experiment.

In [1]:
# Import libraries and the reusable CDM package.

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import context_dependent_model as cdm

## <span style="color:#1F4E79;">Reusable Functions and Classes</span>

Core CDM calculations are imported from `context_dependent_model`.
Paper-specific generation and experiment routines will be added to
`cdm_paper_reproduction`.

| Function or class | Purpose |
|:---|:---|
| `cdm.CDMChoiceData` | Stores choice sets, padding masks and chosen positions. |
| `cdm.center_cdm_parameters()` | Sets diagonal entries to zero and centres off-diagonal interactions. |
| `cdm.cdm_choice_utilities()` | Sums the context contributions for each available item. |
| `cdm.cdm_choice_probabilities()` | Converts utilities into choice probabilities using softmax. |
| `cdm.average_cdm_negative_log_likelihood()` | Computes the mean negative log-probability of observed choices. |
| `cdm.generate_cdm_choice_data_on_sets()` | Samples choices from a supplied interaction matrix and supplied choice sets. |
| `cdm.FullContextDependentModel` | Represents the trainable full CDM interaction matrix. |
| `cdm.optimize_full_cdm_with_adam()` | Fits a full CDM using Adam and returns optimization diagnostics. |
| `cdm.cdm_parameter_recovery_error()` | Computes off-diagonal parameter RMSE after centring both matrices. |

**Reproduction notes**

- The existing optimizer needs support for the paper's initialization procedure.
- The existing recovery metric returns RMSE; the squared-error metric for
  Figure 1(a) will be defined explicitly.
- Centring removes a common-shift ambiguity; it does not by itself guarantee
  identifiability for every collection of choice sets.

In [2]:
# Locate the project root and create directories for reproduction outputs.

PROJECT_ROOT = next(
    (
        directory
        for directory in (Path.cwd(), *Path.cwd().parents)
        if (directory / "pyproject.toml").is_file()
        and (directory / "src").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook inside the thesis repository."
    )

EXPERIMENT_NAME = "01_synthetic_recovery"
EXPERIMENT_PATH = Path("cdm_paper_reproduction") / EXPERIMENT_NAME

FIGURE_DIRECTORY = (
    PROJECT_ROOT / "docs/thesis/images/experiments" / EXPERIMENT_PATH
)

TABLE_DIRECTORY = (
    PROJECT_ROOT / "docs/thesis/tables/experiments" / EXPERIMENT_PATH
)

RESULT_DIRECTORY = (
    PROJECT_ROOT / "results/experiments" / EXPERIMENT_PATH
)

output_directories = {
    "Figures": FIGURE_DIRECTORY,
    "LaTeX tables": TABLE_DIRECTORY,
    "Numerical results": RESULT_DIRECTORY,
}

for directory in output_directories.values():
    directory.mkdir(parents=True, exist_ok=True)

directory_summary = pd.DataFrame(
    [
        {"Output": name, "Directory": str(directory)}
        for name, directory in output_directories.items()
    ]
)

directory_summary

,Output,Directory
0,Figures,/Users/zsres/Documents/M.Tech/Nitesh-Kumar-Sha...
1,LaTeX tables,/Users/zsres/Documents/M.Tech/Nitesh-Kumar-Sha...
2,Numerical results,/Users/zsres/Documents/M.Tech/Nitesh-Kumar-Sha...


In [3]:
# Set consistent typography, colours and display formatting.

plt.style.use("default")

PLOT_COLOURS = {
    "estimate": "#0072B2",
    "reference": "#555555",
    "individual_run": "#A7B5C5",
    "comparison": "#D55E00",
}

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "dejavuserif",
    "font.size": 11,
    "figure.figsize": (8, 5),
    "figure.dpi": 120,
    "figure.facecolor": "white",
    "figure.constrained_layout.use": True,
    "axes.facecolor": "white",
    "axes.titlesize": 13,
    "axes.titleweight": "semibold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.axisbelow": True,
    "axes.grid": True,
    "grid.alpha": 0.20,
    "grid.linewidth": 0.7,
    "grid.linestyle": "--",
    "lines.linewidth": 2,
    "lines.markersize": 6,
    "legend.fontsize": 10,
    "legend.frameon": False,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.08,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
})

pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 20)

print("Plot and table appearance configured.")

Plot and table appearance configured.
